In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
import os

In [75]:
# Cargamos los csv de los tifs
path = "saved_files/dataset"
dfs = {}
for archivo in os.listdir(path):
    if archivo.endswith("_features.csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs[nombre_sin_extension] = pd.read_csv(ruta_completa)

In [76]:
dfs["c2x-complex-nets_1x1_imida_depth_gt_1_features"].head(2)

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,...,dif_rel_4bands_rhown_B3_B4_B5_B1,dif_rel_4bands_rhown_B3_B4_B5_B2,dif_rel_4bands_rhown_B3_B5_B4_B1,dif_rel_4bands_rhown_B3_B5_B4_B2,dif_rel_4bands_rhown_B4_B1_B5_B2,dif_rel_4bands_rhown_B4_B1_B5_B3,dif_rel_4bands_rhown_B4_B2_B5_B1,dif_rel_4bands_rhown_B4_B2_B5_B3,dif_rel_4bands_rhown_B4_B3_B5_B1,dif_rel_4bands_rhown_B4_B3_B5_B2
0,2017-06-30,CTD1,4187246,695025,0.023568,0.034184,0.043664,0.014988,0.009146,0.00232,...,2.434,2.563,3.829,4.033,0.367,0.428,0.033,0.224,-0.063,0.067
1,2017-06-30,CTD2,4181518,693105,0.013951,0.022364,0.026365,0.006800,0.003868,0.00100,...,3.431,3.546,5.623,5.812,0.311,0.339,0.006,0.150,-0.039,0.077


Las fórmulas que queremos comprobar son:

- $y_1 = 0.6386 e^{4.7513x}$, donde $y$ es la clorofila y $x$ es la relación de $(Green - Blue)/(Green + Blue)$

- $y_2 = 124.94x - 115.35$ donde $x = \frac{Green+NIR1}{Green + Red}$

- $y_3 = 22.835x - 12.974$ donde $x = \frac{NIR1 - NIR2}{Red - NIR2}$

- $y_4 = 32.448x - 21.408$ donde $x = \frac{NIR1}{Red}$


con las fechas


dates = [
    "28/10/2016", "20/06/2018", "07/11/2018", "14/08/2019",
    "30/06/2017", "10/07/2018", "12/03/2019", "18/09/2019",
    "20/02/2018", "29/08/2018", "25/06/2019", "03/10/2019",
    "07/03/2018", "03/10/2018"
]


Filtramos los dataframes para dejar solamente esas fechas:

In [77]:
dates = [
    "28/10/2016", "20/06/2018", "07/11/2018", "14/08/2019",
    "30/06/2017", "10/07/2018", "12/03/2019", "18/09/2019",
    "20/02/2018", "29/08/2018", "25/06/2019", "03/10/2019",
    "07/03/2018", "03/10/2018"
]
filter_dates = pd.to_datetime(dates, format="%d/%m/%Y")

for nombre_df, df in dfs.items():
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d", errors='coerce')  # asegurarse
    dfs[nombre_df] = df[df["Date"].isin(filter_dates)].copy()


Recordemos que en el paper quitan las observaciones cuya medida de Chl-a esté fuera de la mediana $\pm$ una desviación estándar. Para ello hacemos lo siguiente (no lo hacemos porque los resultados empeoran muchísimo.). De hacerlo, pasaríamos de tener 130 filas, a quedarnos solamente con 107.

In [69]:
# for nombre_df, df in dfs.items():
#     if "Chl" in df.columns:
#         mediana = df["Chl"].median()
#         desviacion = df["Chl"].std()
#         umbral_sup = mediana + desviacion
#         umbral_inf = mediana - desviacion

#         # Filtrar
#         df_filtrado = df[(df["Chl"] >= umbral_inf) & (df["Chl"] <= umbral_sup)].copy()
#         dfs[nombre_df] = df_filtrado

Definimos funciones para las fórmulas que queremos aplicar.
Recordar que:
- Blue = rhow_B2
- Green = rhow_B3
- Red = rhow_B4
- NIR1 = rhow_B5
- NIR2 = rhow_B6

In [78]:
def add_y(blue, green, red, nir1, nir2):
    x_1 = (green-blue)/(green+blue)
    y_1 = 0.6386*np.exp(4.7513*x_1)

    x_2 = (green+nir1)/(green+red)
    y_2 = 124.94*x_2 - 115.35

    x_3 = (nir1-nir2)/(red-nir2)
    y_3 = 22.835*x_3 - 12.974

    x_4 = nir1/red
    y_4 = 32.448*x_4 - 21.408

    return y_1, y_2, y_3, y_4

In [79]:
def add_y_columns_to_df(df, band_set="rhow"):
    # Mapas de bandas
    mapping = band_maps[band_set]

    # Extraer bandas desde el DataFrame
    blue = df[mapping["Blue"]]
    green = df[mapping["Green"]]
    red = df[mapping["Red"]]
    nir1 = df[mapping["NIR1"]]
    nir2 = df[mapping["NIR2"]]

    # Aplicar fórmula
    y_1, y_2, y_3, y_4 = add_y(blue, green, red, nir1, nir2)

    # Añadir columnas al DataFrame
    if band_set == "rhow":
        df["y_1_rhow"] = y_1
        df["y_2_rhow"] = y_2
        df["y_3_rhow"] = y_3
        df["y_4_rhow"] = y_4

    elif band_set == "rhown":
        df["y_1_rhown"] = y_1
        df["y_2_rhown"] = y_2
        df["y_3_rhown"] = y_3
        df["y_4_rhown"] = y_4

    return df

band_maps = {
    "rhow": {
        "Blue": "rhow_B2",
        "Green": "rhow_B3",
        "Red": "rhow_B4",
        "NIR1": "rhow_B5",
        "NIR2": "rhow_B6"
    },
    "rhown": {
        "Blue": "rhown_B2",
        "Green": "rhown_B3",
        "Red": "rhown_B4",
        "NIR1": "rhown_B5",
        "NIR2": "rhown_B6"
    }
}

In [80]:
for nombre_df, df in dfs.items():
    # De momento solo con rhow
    for band_set in ["rhow", "rhown"]:
        dfs[nombre_df] = add_y_columns_to_df(df, band_set)

In [111]:
metrics_df = pd.DataFrame()
for nombre_df, df in dfs.items():

    #R2
    # try:
    #     metrics_df.loc[nombre_df[:-9], "r2_y_1_rhow"] = r2_score(df["Chl"], df["y_1_rhow"]).round(3)
    #     metrics_df.loc[nombre_df[:-9], "r2_y_2_rhow"] = r2_score(df["Chl"], df["y_2_rhow"]).round(3)
    #     metrics_df.loc[nombre_df[:-9], "r2_y_3_rhow"] = r2_score(df["Chl"], df["y_3_rhow"]).round(3)
    #     metrics_df.loc[nombre_df[:-9], "r2_y_4_rhow"] = r2_score(df["Chl"], df["y_4_rhow"]).round(3)
    # except:
    #     continue

    # try:
    #     metrics_df.loc[nombre_df[:-9], "r2_y_1_rhown"] = r2_score(df["Chl"], df["y_1_rhown"]).round(3)
    #     metrics_df.loc[nombre_df[:-9], "r2_y_2_rhown"] = r2_score(df["Chl"], df["y_2_rhown"]).round(3)
    #     metrics_df.loc[nombre_df[:-9], "r2_y_3_rhown"] = r2_score(df["Chl"], df["y_3_rhown"]).round(3)
    #     metrics_df.loc[nombre_df[:-9], "r2_y_4_rhown"] = r2_score(df["Chl"], df["y_4_rhown"]).round(3)
    # except:
    #     continue

    # RMSE
    try:
        metrics_df.loc[nombre_df[:-9], "rmse_y_1_rhow"] = np.sqrt(mean_squared_error(df["Chl"], df["y_1_rhow"])).round(3)
        metrics_df.loc[nombre_df[:-9], "rmse_y_2_rhow"] = np.sqrt(mean_squared_error(df["Chl"], df["y_2_rhow"])).round(3)
        metrics_df.loc[nombre_df[:-9], "rmse_y_3_rhow"] = np.sqrt(mean_squared_error(df["Chl"], df["y_3_rhow"])).round(3)
        metrics_df.loc[nombre_df[:-9], "rmse_y_4_rhow"] = np.sqrt(mean_squared_error(df["Chl"], df["y_4_rhow"])).round(3)
    except:
        continue

    try:
        metrics_df.loc[nombre_df[:-9], "rmse_y_1_rhown"] = np.sqrt(mean_squared_error(df["Chl"], df["y_1_rhown"])).round(3)
        metrics_df.loc[nombre_df[:-9], "rmse_y_2_rhown"] = np.sqrt(mean_squared_error(df["Chl"], df["y_2_rhown"])).round(3)
        metrics_df.loc[nombre_df[:-9], "rmse_y_3_rhown"] = np.sqrt(mean_squared_error(df["Chl"], df["y_3_rhown"])).round(3)
        metrics_df.loc[nombre_df[:-9], "rmse_y_4_rhown"] = np.sqrt(mean_squared_error(df["Chl"], df["y_4_rhown"])).round(3)
    except:
        continue

    #NRMSE
    try:
        metrics_df.loc[nombre_df[:-9], "nrmse_y_1_rhow"] = (metrics_df.loc[nombre_df[:-9], "rmse_y_1_rhow"]/(df["Chl"].max()-df["Chl"].min())).round(3)*100
        metrics_df.loc[nombre_df[:-9], "nrmse_y_2_rhow"] = (metrics_df.loc[nombre_df[:-9], "rmse_y_2_rhow"]/(df["Chl"].max()-df["Chl"].min())).round(3)*100
        metrics_df.loc[nombre_df[:-9], "nrmse_y_3_rhow"] = (metrics_df.loc[nombre_df[:-9], "rmse_y_3_rhow"]/(df["Chl"].max()-df["Chl"].min())).round(3)*100
        metrics_df.loc[nombre_df[:-9], "nrmse_y_4_rhow"] = (metrics_df.loc[nombre_df[:-9], "rmse_y_4_rhow"]/(df["Chl"].max()-df["Chl"].min())).round(3)*100
    except:
        continue

In [114]:
metrics_df[metrics_df.lt(2.5).any(axis=1)]

,rmse_y_1_rhow,rmse_y_2_rhow,rmse_y_3_rhow,rmse_y_4_rhow,rmse_y_1_rhown,rmse_y_2_rhown,rmse_y_3_rhown,rmse_y_4_rhown,nrmse_y_1_rhow,nrmse_y_2_rhow,nrmse_y_3_rhow,nrmse_y_4_rhow
c2x-nets_1x1_upct_depth_lt_1,3.766,3.015,2.658,2.967,3.611,2.673,2.424,2.566,20.8,16.6,14.7,16.4
c2x-nets_9x9_upct_depth_lt_2,4.260,2.758,2.431,2.683,4.149,3.053,2.782,2.768,19.5,12.6,11.1,12.3
c2x-nets_5x5_imida_depth_gt_1,2.624,4.289,3.816,4.011,2.553,2.850,2.498,2.727,18.1,29.6,26.4,27.7
c2x-nets_5x5_upct_depth_lt_2,4.226,2.717,2.332,2.638,4.091,2.946,2.652,2.647,19.3,12.4,10.7,12.1
c2x-complex-nets_5x5_upct_depth_lt_1,4.000,7.114,6.493,6.121,3.940,3.081,2.384,2.403,22.1,39.3,35.8,33.8
c2x-nets_9x9_imida_depth_gt_1,2.550,4.265,3.808,3.995,2.489,2.863,2.543,2.769,17.6,29.5,26.3,27.6
c2x-nets_5x5_upct_depth_gt_1,3.178,3.564,3.096,3.352,3.073,2.697,2.322,2.474,15.3,17.1,14.9,16.1
c2x-complex-nets_3x3_upct_depth_lt_1,4.010,7.068,6.221,6.017,3.953,3.027,2.346,2.422,22.1,39.0,34.3,33.2
c2x-complex-nets_9x9_upct_depth_lt_1,3.986,7.113,7.082,6.328,3.913,3.044,2.497,2.518,22.0,39.3,39.1,34.9
c2x-nets_3x3_upct_depth_lt_2,4.244,2.843,2.421,2.743,4.106,2.932,2.581,2.637,19.4,13.0,11.1,12.5


In [100]:
metrics_df[metrics_df.gt(0.65).any(axis=1)]

,r2_y_1_rhow,r2_y_2_rhow,r2_y_3_rhow,r2_y_4_rhow,r2_y_1_rhown,r2_y_2_rhown,r2_y_3_rhown,r2_y_4_rhown,rmse_y_1_rhow,rmse_y_2_rhow,rmse_y_3_rhow,rmse_y_4_rhow,rmse_y_1_rhown,rmse_y_2_rhown,rmse_y_3_rhown,rmse_y_4_rhown
c2x-nets_1x1_imida_depth_lt_2,0.191,0.397,0.519,0.431,0.245,0.472,0.571,0.541,4.074,3.516,3.140,3.417,3.935,3.291,2.966,3.069
c2x-complex-nets_3x3_imida_depth_lt_2,-0.036,-1.343,-0.743,-0.600,-0.005,0.337,0.612,0.600,4.609,6.932,5.978,5.727,4.540,3.688,2.821,2.862
c2x-nets_1x1_upct_depth_lt_2,0.221,0.629,0.713,0.634,0.276,0.579,0.654,0.639,4.283,2.954,2.599,2.936,4.129,3.147,2.855,2.917
c2x-complex-nets_5x5_imida_depth_gt_1,-0.094,-6.114,-4.865,-4.082,-0.054,-0.565,0.157,0.056,3.132,7.988,7.253,6.752,3.075,3.746,2.749,2.910
c2x-nets_1x1_imida_depth_gt_1,0.222,-1.320,-0.899,-1.013,0.286,-0.061,0.164,0.014,2.642,4.562,4.127,4.250,2.531,3.084,2.738,2.975
c2x-complex-nets_3x3_imida_depth_lt_1,-0.026,-0.466,-0.087,-0.025,-0.002,0.407,0.591,0.597,5.624,6.720,5.786,5.620,5.556,4.275,3.548,3.523
c2x-nets_1x1_upct_depth_lt_1,0.198,0.486,0.600,0.502,0.262,0.596,0.668,0.627,3.766,3.015,2.658,2.967,3.611,2.673,2.424,2.566
c2x-complex-nets_3x3_upct_depth_lt_2,0.043,-0.972,-0.477,-0.380,0.069,0.478,0.681,0.680,4.746,6.815,5.897,5.701,4.682,3.505,2.739,2.743
c2x-nets_9x9_upct_depth_lt_2,0.229,0.677,0.749,0.694,0.269,0.604,0.671,0.674,4.260,2.758,2.431,2.683,4.149,3.053,2.782,2.768
c2x-complex-nets_1x1_upct_depth_gt_1,-0.035,-3.852,-2.982,-2.228,0.004,-0.097,0.533,0.495,3.756,8.133,7.367,6.633,3.685,3.867,2.524,2.624
